# 第3回実習：UR5e＋Robotiq 2F-85 の装着・開閉（MuJoCo / Colab）

**目的**：既存の第2回 `02_ur5e_ai_mujoco_colab.ipynb` を**一切変更せず**、公式UR5eの先端に公式Robotiq 2F-85モデルを装着し、「開→閉→開」を物理シミュレーションで確認する。

**実習の範囲**

- 実施：UR5eと2F-85のモデル結合、6軸アームをHOME姿勢に保持、指の開閉、拡大映像と全体映像の記録、関節角・制御入力のCSV記録
- **未実装**：実物体の把持・搬送、AIによる作業指示の解釈、ROS 2への接続、実機の安全性検証
- コードセルは上から順番に実行。最初は**セル1～3（モデル統合・静止画像）**まで試し、そこで3Dモデルの装着位置が妥当か確認する。次にセル4で動画を作成する。
- 前回の`02`や公開済みの65秒動画は上書きしない。出力名は `ur5e_2f85_open_close_test` 系列に統一。

**使用モデル**：Google DeepMind *MuJoCo Menagerie* 公式 `universal_robots_ur5e`（BSD-3-Clause）、`robotiq_2f85`（BSD-2-Clause）。Menagerieの`attachment_site`を使い、MuJoCoの`MjSpec.attach()`で一体化する。モデルは初回のみネット経由で取得。

**重要**：このノートブックはPythonコードの構文・ノートブック形式を確認した教材です。制作環境からColab/MuJoCo本体へのネット接続ができず、**統合モデルのコンパイルと物理シミュレーションの実動作は未検証**です。エラー時は該当セルの赤いエラー表示を共有してください。

## 0. 実習の構造

```text
UR5e (MuJoCo Menagerie) ─┐
                        ├─ MjSpec.attach ─ UR5e + Robotiq 2F-85
Robotiq 2F-85 ──────────┘                 │
                                          ├─ アーム6軸：HOME姿勢保持
                                          └─ 指：制御入力 0 ⇄ 255
                                                   │
                                               MP4 + CSV
```

Robotiqの制御入力の0～255はモデル内部の**無次元制御値**であり、開口幅[mm]でも力[N]でもありません。0は開方向、255は閉方向に対応します。

## 1. セル1：無料Colab環境へのインストール

右上の「接続」でCPUランタイムを開始し、以下のセルの三角ボタンを押してください。前のノートブックとは独立した環境として使います。

In [ ]:
# セル1：必要なライブラリ。初回は数分かかることがあります。
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y libosmesa6 libosmesa6-dev fonts-noto-cjk > /dev/null
%pip -q install "mujoco>=3.3,<4" "mujoco-menagerie==2026.10.3" "imageio[ffmpeg]" pillow pandas numpy
print("セル1完了：インストール成功")

## 2. セル2：UR5eと2F-85のモデルを一体化

Menagerie公開モデルを読み込み、UR5eの`attachment_site`にグリッパー全体を結合します。**ここではまだ動画は生成しません。** モデル内にアーム6軸とグリッパー制御1系統が存在することを確認します。

In [ ]:
# セル2：モデル接続と初期姿勢の設定
import os
os.environ["MUJOCO_GL"] = "osmesa"   # mujocoをimportする前に指定

import mujoco
import mujoco_menagerie as mm
import numpy as np
import pandas as pd
from pathlib import Path

print("MuJoCo version:", mujoco.__version__)

# 公式MJCFを「編集可能なMjSpec」として取得
ur5e_spec = mm.get("universal_robots_ur5e").spec("scene")
gripper_spec = mm.get("robotiq_2f85").spec("2f85")

# UR5e公式モデルにはattachment_siteが定義されている
mount_site = ur5e_spec.site("attachment_site")
assert mount_site is not None, "UR5eのattachment_siteが見つかりません。"

# 接続先：UR5e wrist_3_linkのattachment_site
# prefixにより他のリンク名・材料名・アクチュエータ名との重複を防ぐ
ur5e_spec.attach(gripper_spec, prefix="grip_", site=mount_site)

# 結合モデルをコンパイル
model = ur5e_spec.compile()
data = mujoco.MjData(model)

# 結合モデルは「アーム6関節 + グリッパー8関節」、制御は「アーム6 + グリッパー1」の想定
assert model.nu == 7, f"制御アクチュエータの数が想定と異なります: nu={model.nu}"
assert model.nq >= 14, f"関節自由度が想定と異なります: nq={model.nq}"

ARM_JOINTS = [
    "shoulder_pan_joint", "shoulder_lift_joint", "elbow_joint",
    "wrist_1_joint", "wrist_2_joint", "wrist_3_joint",
]
ARM_ACTUATORS = [
    "shoulder_pan", "shoulder_lift", "elbow",
    "wrist_1", "wrist_2", "wrist_3",
]
HOME_Q = np.array([-1.5708, -1.5708, 1.5708, -1.5708, -1.5708, 0.0])

# 文字列名からMuJoCo内部の番号を取得。番号の並び順を暗黙に仮定しない。
def lookup(obj_type, name):
    number = mujoco.mj_name2id(model, obj_type, name)
    if number < 0:
        raise ValueError(f"モデル内に {name!r} がありません。アクチュエータ名を確認してください。")
    return int(number)

arm_joint_ids = [lookup(mujoco.mjtObj.mjOBJ_JOINT, n) for n in ARM_JOINTS]
arm_qpos_ids = [int(model.jnt_qposadr[i]) for i in arm_joint_ids]
arm_actuator_ids = [lookup(mujoco.mjtObj.mjOBJ_ACTUATOR, n) for n in ARM_ACTUATORS]

finger_actuator_id = lookup(mujoco.mjtObj.mjOBJ_ACTUATOR, "grip_fingers_actuator")
left_joint_id = lookup(mujoco.mjtObj.mjOBJ_JOINT, "grip_left_driver_joint")
right_joint_id = lookup(mujoco.mjtObj.mjOBJ_JOINT, "grip_right_driver_joint")
left_qpos_id = int(model.jnt_qposadr[left_joint_id])
right_qpos_id = int(model.jnt_qposadr[right_joint_id])
pinch_site_id = lookup(mujoco.mjtObj.mjOBJ_SITE, "grip_pinch")

# UR5eをHOME姿勢に設定。グリッパーは全開(制御入力0)から開始。
mujoco.mj_resetData(model, data)
for qid, target in zip(arm_qpos_ids, HOME_Q):
    data.qpos[qid] = target
for aid, target in zip(arm_actuator_ids, HOME_Q):
    data.ctrl[aid] = target
data.ctrl[finger_actuator_id] = 0.0
mujoco.mj_forward(model, data)

print(f"モデル統合成功: nq={model.nq}, nv={model.nv}, nu={model.nu}")
print("グリッパー制御アクチュエータ:", mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, finger_actuator_id))
print("制御範囲:", model.actuator_ctrlrange[finger_actuator_id].tolist())
print("グリッパーの指関節位置(初期):", float(data.qpos[left_qpos_id]), float(data.qpos[right_qpos_id]))
print("先端把持点の位置[m]:", np.round(data.site_xpos[pinch_site_id], 4))
print("★ 次はセル3の静止画像を確認してください。")

## 3. セル3：装着位置を静止画像で確認

全体図とグリッパー拡大図を表示します。**この画像が成功するまではセル4へ進まないでください。** グリッパーが先端に付いていない場合はモデル接続の問題です。

In [ ]:
# セル3：UR5e全体 / グリッパー拡大の2画面
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Video
import imageio.v2 as imageio

W, H = 480, 360
renderer = mujoco.Renderer(model, height=H, width=W)

whole_camera = mujoco.MjvCamera()
whole_camera.type = mujoco.mjtCamera.mjCAMERA_FREE
whole_camera.azimuth = 135
whole_camera.elevation = -24
whole_camera.distance = 1.70
whole_camera.lookat[:] = [0, 0, 0.35]

close_camera = mujoco.MjvCamera()
close_camera.type = mujoco.mjtCamera.mjCAMERA_FREE
close_camera.azimuth = 135
close_camera.elevation = -15
close_camera.distance = 0.34

font_file = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font = ImageFont.truetype(font_file, 18) if Path(font_file).exists() else ImageFont.load_default()
small_font = ImageFont.truetype(font_file, 15) if Path(font_file).exists() else ImageFont.load_default()


def render_pair(phase="OPEN", cmd=0):
    """実機ではなく、MuJoCoの現在状態を2視点で撮影する。"""
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera=whole_camera)
    view1 = renderer.render().copy()
    close_camera.lookat[:] = data.site_xpos[pinch_site_id]
    renderer.update_scene(data, camera=close_camera)
    view2 = renderer.render().copy()
    merged = Image.new("RGB", (2 * W, H + 64), (16, 27, 41))
    merged.paste(Image.fromarray(view1), (0, 64))
    merged.paste(Image.fromarray(view2), (W, 64))
    draw = ImageDraw.Draw(merged)
    draw.text((12, 5), f"UR5e + Robotiq 2F-85 | {phase}", font=font, fill="white")
    draw.text((12, 35), "全体 (left)", font=small_font, fill=(177, 216, 235))
    draw.text((W + 12, 35), f"指先拡大 (right) | ctrl={cmd}/255", font=small_font, fill=(177, 216, 235))
    return np.asarray(merged)

preview = render_pair()
display(Image.fromarray(preview))
print("装着・表示の確認用画像です。グリッパーの向き、先端との接合を確認してください。")

## 4. セル4：2F-85を開→閉→開し、MP4とCSVを作る

アームはHOME姿勢に保持し、グリッパーのアクチュエータ指令を段階的に変更します。

- **OPEN**：`data.ctrl[finger_actuator_id] = 0`
- **CLOSE**：`data.ctrl[finger_actuator_id] = 255`
- **OPEN**：再び0

これは指令値による開閉試験であり、把持力・実測開口幅や実物体の搬送を評価するものではありません。動作の変化を左右の駆動関節角で検査します。

In [ ]:
# セル4：開閉動画とログの作成
# 既存の02とは異なるファイル名で保存し、過去の成果を上書きしない。
from IPython.display import Video, display
import time

OUTPUT_VIDEO = "ur5e_2f85_open_close_test.mp4"
OUTPUT_CSV = "ur5e_2f85_open_close_test_log.csv"
FPS = 8
PLAN = [("OPEN", 0.0, 3.0), ("CLOSE", 255.0, 4.0), ("OPEN", 0.0, 4.0)]

# 開閉前の初期状態から再現可能な形でスタート
mujoco.mj_resetData(model, data)
for qid, target in zip(arm_qpos_ids, HOME_Q):
    data.qpos[qid] = target
for aid, target in zip(arm_actuator_ids, HOME_Q):
    data.ctrl[aid] = target
data.ctrl[finger_actuator_id] = 0.0
mujoco.mj_forward(model, data)

log_rows = []
frames_written = 0
control_dt = float(model.opt.timestep)
if control_dt <= 0:
    raise ValueError(f"物理刻み時間が不正です: {control_dt}")

with imageio.get_writer(OUTPUT_VIDEO, fps=FPS, codec="libx264", quality=7, macro_block_size=1) as video:
    for phase, command, duration_s in PLAN:
        data.ctrl[finger_actuator_id] = command
        count_frames = int(round(duration_s * FPS))
        for _ in range(count_frames):
            # 物理シミュレーションの時刻が必要なフレーム時刻に達するまで進める
            desired_sim_time = (frames_written + 1) / FPS
            while data.time + control_dt * 0.5 < desired_sim_time:
                mujoco.mj_step(model, data)
            if not np.all(np.isfinite(data.qpos)):
                raise RuntimeError("関節位置にNaN/Infを検出。シミュレーションを停止します。")

            frame = render_pair(phase=phase, cmd=int(command))
            video.append_data(frame)
            arm_error = float(np.max(np.abs(data.qpos[arm_qpos_ids] - HOME_Q)))
            log_rows.append({
                "time_s": round(float(data.time), 4),
                "phase": phase,
                "gripper_ctrl_0_255": int(command),
                "left_driver_rad": float(data.qpos[left_qpos_id]),
                "right_driver_rad": float(data.qpos[right_qpos_id]),
                "max_ur5e_home_error_rad": arm_error,
                "contacts_n": int(data.ncon),
            })
            frames_written += 1

log_df = pd.DataFrame(log_rows)
log_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"動画生成完了: {OUTPUT_VIDEO}, {frames_written/FPS:.1f} 秒 / {frames_written} フレーム")
print("CSV保存完了:", OUTPUT_CSV)
print(log_df.groupby('phase', sort=False)[['gripper_ctrl_0_255', 'left_driver_rad', 'right_driver_rad']].agg(['min','max']).to_string())
print("左右駆動関節の実測変化量 [rad]:")
print("left:", round(float(log_df.left_driver_rad.max()-log_df.left_driver_rad.min()),4))
print("right:", round(float(log_df.right_driver_rad.max()-log_df.right_driver_rad.min()),4))

if (log_df.left_driver_rad.max() - log_df.left_driver_rad.min()) < 0.10:
    print("注意：左指の移動が小さいため、グリッパーの制御・結合状態を再確認してください。")
else:
    print("開閉動作による左駆動関節の変化を検出しました。")
display(Video(OUTPUT_VIDEO, embed=True, width=960))

## 5. セル5：実行結果をグラフで確認（任意）

制御入力0→255→0と、左右の駆動関節角の追従を区別して示します。グラフが実際に開閉しているかの確認に有用です。

In [ ]:
# セル5：実測関節角と入力指令を確認するグラフ
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(log_df['time_s'], log_df['left_driver_rad'], label='left driver [rad]')
ax.plot(log_df['time_s'], log_df['right_driver_rad'], label='right driver [rad]')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Joint angle [rad]')
ax.grid(True, alpha=0.25)
ax2 = ax.twinx()
ax2.step(log_df['time_s'], log_df['gripper_ctrl_0_255'], where='post', linestyle='--', label='Command [0-255]', alpha=0.6)
ax2.set_ylabel('Actuator command [0-255]')
ax2.set_ylim(-10, 270)
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('Robotiq 2F-85: command vs actual joint angles')
plt.show()

## 6. セル6：PCに動画とCSVをダウンロード

セル4が成功してから実行します。ブラウザが「複数ファイルのダウンロード許可」を求めた場合は許可してください。

In [ ]:
# セル6：ダウンロード（Colab用）
from google.colab import files
files.download(OUTPUT_VIDEO)
files.download(OUTPUT_CSV)

## 7. 成功判定と次の研究課題

**第3回の成功条件**

1. セル2で`nu=7`（アーム6＋グリッパー1）が表示される。
2. セル3に、UR5e先端へ装着されたRobotiq 2F-85が**全体図と拡大図**で確認できる。
3. セル4の動画で「OPEN→CLOSE→OPEN」の指が実際に変化する。
4. CSVの左右ドライバ関節角に明確な変化がある。制御入力0～255は開口幅[mm]ではない。
5. 02ノートブック、既存の65秒YouTube動画を変更していない。

**次段階（まだ実装しない）**：第2回の文章AI推論結果をロボットとグリッパー指令に連携し、物体の位置認識、接触把持、Pick & Placeへ進む。グリッパー装着時にはエンドエフェクタの質量・慣性・TCPの変更があるため、過去の関節目標到達条件や可搬質量をそのまま実機に適用しない。

**GitHubに登録する場合**：既存の `ur5e-physical-ai-2026` リポジトリを使用し、`03_ur5e_robotiq_2f85_colab.ipynb` とログCSVを追加する。まだ開発中であること、ROS 2未使用、物体把持未実装であることをREADMEに明記する。GitHubへMP4をアップロードする必要はない。

**技術的な参考情報**

- [Google DeepMind, MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie)
- [UR5e MJCF（attachment_siteの定義）](https://github.com/google-deepmind/mujoco_menagerie/blob/main/universal_robots_ur5e/ur5e.xml)
- [Robotiq 2F-85 MJCF（指アクチュエータと制御範囲）](https://github.com/google-deepmind/mujoco_menagerie/blob/main/robotiq_2f85/2f85.xml)
- [MuJoCoモデル編集・モデル結合（MjSpec）](https://mujoco.readthedocs.io/en/stable/programming/modeledit.html)
- [MuJoCo Menagerie FAQ：グリッパーの接続](https://github.com/google-deepmind/mujoco_menagerie/blob/main/FAQ.md)